# 05 Embedder Training

Goal: train the first style embedder with triplet loss. All model training for this project happens in notebooks; package code only provides reusable architecture/data pieces.

Important: `triplets_dry_run.jsonl` checks mechanics only. Meaningful learning requires real neutral drafts.

If dependencies are missing, run this once:

```python
import sys, subprocess
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
req = next((p / "requirements.txt" for p in candidates if (p / "requirements.txt").exists()), None)
cmd = [sys.executable, "-m", "pip", "install"]
cmd += ["-r", str(req)] if req else ["torch", "transformers", "openai", "python-dotenv"]
subprocess.check_call(cmd)
```

In [ ]:
import os
import random
import sys
from collections import defaultdict
from itertools import cycle
from pathlib import Path

def find_project_root():
    env_root = os.getenv("VOICE_CLONE_ROOT")
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    for candidate in candidates:
        if (candidate / "voice_clone").exists():
            return candidate
    raise RuntimeError("Could not find project root. Set VOICE_CLONE_ROOT to the repo path in this kernel.")

ROOT = find_project_root()
sys.path.insert(0, str(ROOT))

import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from voice_clone.data.dataset import TripletTextDataset
from voice_clone.models import StyleEmbedder, count_parameters

print("ROOT:", ROOT)

In [ ]:
# Core config. Start small, then scale.
# Generate this with: scripts/build_data_pipeline.py style-regen ...
# Use triplets_dry_run.jsonl only for mechanics checks.
TRIPLETS_PATH = ROOT / "data" / "processed" / "triplets_style_regen_small.jsonl"
MODEL_NAME = "allenai/longformer-base-4096"
CHECKPOINT_DIR = ROOT / "checkpoints"

SEED = 7
BATCH_SIZE = 1
MAX_LENGTH = 512
STEPS = 50
LR = 1e-5
MARGIN = 0.25
GRAD_ACCUM_STEPS = 4
LOG_EVERY = 5
EVAL_EVERY = 25
EVAL_AUTHORS = 8
EVAL_PASSAGES_PER_AUTHOR = 2
GRADIENT_CHECKPOINTING = True

random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)
print("triplets:", TRIPLETS_PATH)

In [ ]:
dataset = TripletTextDataset.from_jsonl(TRIPLETS_PATH, seed=SEED)
sample = dataset[0]

if sample["negative"].startswith("[DRY RUN"):
    print("WARNING: dry-run negatives are placeholders. This verifies training mechanics, not final learning quality.")
print("negative type:", sample.get("negative_type"))

def collate_triplets(batch):
    return {
        "anchor": [item["anchor"] for item in batch],
        "positive": [item["positive"] for item in batch],
        "negative": [item["negative"] for item in batch],
        "author_id": [item["author_id"] for item in batch],
    }

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_triplets)

print("dataset size:", len(dataset))
print("sample author:", sample["author_id"])
print("anchor doc:", sample["anchor_doc_id"])
print("positive doc:", sample["positive_doc_id"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = StyleEmbedder.from_pretrained(
    MODEL_NAME,
    gradient_checkpointing=GRADIENT_CHECKPOINTING,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
print(count_parameters(model))

In [ ]:
def encode_texts(texts):
    encoded = tokenizer(
        list(texts),
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    return {key: value.to(device) for key, value in encoded.items()}


def triplet_cosine_loss(anchor, positive, negative, margin=MARGIN):
    positive_distance = 1 - torch.sum(anchor * positive, dim=-1)
    negative_distance = 1 - torch.sum(anchor * negative, dim=-1)
    return F.relu(positive_distance - negative_distance + margin).mean()


def embed_texts(texts, batch_size=4):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            encoded = encode_texts(texts[start:start + batch_size])
            outputs.append(model(**encoded).cpu())
    return torch.cat(outputs, dim=0)

In [ ]:
def build_eval_records(records, authors=EVAL_AUTHORS, passages_per_author=EVAL_PASSAGES_PER_AUTHOR):
    grouped = defaultdict(list)
    for record in records:
        grouped[record.author_id].append(record)
    available = max((len(rows) for rows in grouped.values()), default=0)
    passages_per_author = min(passages_per_author, available)
    eligible = [author for author, rows in grouped.items() if len(rows) >= passages_per_author]
    eligible = sorted(eligible)[:authors]

    eval_records = []
    for author in eligible:
        eval_records.extend(grouped[author][:passages_per_author])
    return eval_records


def author_retrieval_eval(records):
    eval_records = build_eval_records(records)
    if not eval_records:
        return {"top1_author_acc": 0.0, "eval_items": 0, "eval_authors": 0}
    texts = [record.real_passage for record in eval_records]
    authors = [record.author_id for record in eval_records]
    embeddings = embed_texts(texts, batch_size=2)
    sims = embeddings @ embeddings.T
    sims.fill_diagonal_(-1e9)
    top_indices = sims.argmax(dim=1).tolist()
    correct = sum(authors[i] == authors[j] for i, j in enumerate(top_indices))
    return {
        "top1_author_acc": correct / len(authors),
        "eval_items": len(authors),
        "eval_authors": len(set(authors)),
    }


author_retrieval_eval(dataset.records)

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
best_acc = -1.0
train_iter = cycle(loader)
optimizer.zero_grad(set_to_none=True)

for step in range(1, STEPS + 1):
    model.train()
    batch = next(train_iter)
    texts = batch["anchor"] + batch["positive"] + batch["negative"]
    encoded = encode_texts(texts)

    embeddings = model(**encoded)
    anchor, positive, negative = embeddings.chunk(3, dim=0)
    loss = triplet_cosine_loss(anchor, positive, negative)
    (loss / GRAD_ACCUM_STEPS).backward()

    if step % GRAD_ACCUM_STEPS == 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if step % LOG_EVERY == 0 or step == 1:
        with torch.no_grad():
            pos_sim = torch.sum(anchor * positive, dim=-1).mean().item()
            neg_sim = torch.sum(anchor * negative, dim=-1).mean().item()
        print(f"step={step:04d} loss={loss.item():.4f} pos_sim={pos_sim:.4f} neg_sim={neg_sim:.4f}")

    if step % EVAL_EVERY == 0 or step == STEPS:
        metrics = author_retrieval_eval(dataset.records)
        print("eval:", metrics)
        if metrics["top1_author_acc"] > best_acc:
            best_acc = metrics["top1_author_acc"]
            checkpoint_path = CHECKPOINT_DIR / "style_embedder_stage3_best.pt"
            torch.save(
                {
                    "model_name": MODEL_NAME,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "step": step,
                    "metrics": metrics,
                    "config": {
                        "max_length": MAX_LENGTH,
                        "margin": MARGIN,
                        "projection_dim": 512,
                    },
                },
                checkpoint_path,
            )
            print("saved:", checkpoint_path)